# Phase 3 — Step 1: verified benchmark inputs

Load the existing S8 publication, inspect six visual variants, and keep model inputs separate from evaluator evidence and provenance. **This notebook does not generate images, author clinical answers, assign splits, or run a benchmark.**

GitHub is the authority for metadata. `DATA_REF="main"` is resolved once to the real current commit; every consumed file comes from that commit. Original JPEGs and existing S8 TAR archives are read from the configured mount. Large metadata already published as `.bin` parts are joined byte for byte and hash-checked; no processing result is regenerated.

Run cells in order. The implementation must be available in your checkout; in Colab, bootstrap it below. Before this PR is merged, set `CODE_REF` to `codex/phase3-benchmark-bootstrap`.

## 1. Load the implementation

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "bogdanparvu18/msc-graduate-project"
CODE_REF = "main"  # Before merge: "codex/phase3-benchmark-bootstrap"
DATA_REF = "main"  # Resolved live; use a saved source_snapshot resolved_commit to reproduce.

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "mmvqa/phase3/storage.py").is_file()), None)
if PROJECT is None:
    PROJECT = Path(tempfile.mkdtemp(prefix="mmvqa-phase3-code-"))
    def git(*args):
        subprocess.run(["git", "-C", str(PROJECT), *args], check=True)
    git("init")
    git("remote", "add", "origin", f"https://github.com/{REPOSITORY}.git")
    git("sparse-checkout", "init", "--cone")
    git("sparse-checkout", "set", "mmvqa", "configs/phase3")
    git("fetch", "--depth", "1", "--filter=blob:none", "origin", CODE_REF)
    git("checkout", "--detach", "FETCH_HEAD")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(PROJECT))
print("Implementation commit:", CODE_COMMIT)


## 2. Configure sources and resolve the current GitHub commit

In [ ]:
from mmvqa.phase3.storage import S8Store
from mmvqa.phase3.benchmark import S8Benchmark, DEFAULT_VARIANTS
from mmvqa.phase3.images import ImageResolver
from mmvqa.phase3.cli import write_json

WORK = Path("/content/mmvqa_phase3") if Path("/content").is_dir() else PROJECT / "outputs/phase3/local/notebook"
STORAGE_ROOT = Path("/content/drive/MyDrive/MMVQA_Clinical")
ORIGINAL_ROOTS = [STORAGE_ROOT]
ARCHIVE_ROOT = STORAGE_ROOT / "outputs/phase2/sector8"
DERIVED_ROOTS = []  # Optional existing S8 run directories; never regenerated here.
VARIANTS = list(DEFAULT_VARIANTS)
PER_VARIANT = 1
PREVIEW_QUESTION = "Describe the visible image content. This is an input-loading preview, not a validated benchmark question."

store = S8Store(repo=REPOSITORY, ref=DATA_REF, cache_dir=WORK / "cache",
                token=os.environ.get("GITHUB_TOKEN"))
print("Requested data ref:", DATA_REF)
print("Resolved data commit:", store.resolve())
benchmark = S8Benchmark(store)
print("Selected processing signature:", benchmark.catalog["processing_signature"])
print("S8 upstream commit is lineage only:", benchmark.catalog.get("upstream_commit"))


## 3. Inspect the selected examples before reading pixels

The default sample uses the **same subject** across `full`, `crop`, `overlay`, `fullcrop`, `temporal_multi`, and `temporal_collage`. Availability does not establish clinical task equivalence. Crops and overlays are annotation-assisted and retain that flag in provenance.

In [ ]:
examples = benchmark.sample(VARIANTS, PER_VARIANT)
source_records = benchmark.source_records(examples)
metadata_previews = [benchmark.inspect(example) for example in examples]
for example in examples:
    print(example["variant"], "images:", example["image_count"],
          "annotation_assisted:", example["annotation_assisted"])
    for item in example["ordered_inputs"]:
        print(" ", item["storage"], item["relative_path"], "sha256:", item["sha256"])
write_json(WORK / "source_snapshot.json", store.snapshot())
print("Metadata verified; image bytes have not been checked yet.")


## 4. Mount the existing storage and load the selected images

In Colab, mount Drive below. On a workstation, change `STORAGE_ROOT`/`ORIGINAL_ROOTS`/`ARCHIVE_ROOT` above to the existing mount. The resolver reads exact paths, verifies hashes and dimensions, and opens only declared regular TAR members. It does not extract or overwrite the dataset. The first use of an archive verifies the whole TAR (the temporal-collage archive is about 3 GB), so that check can take time over Drive. Repeated use in the same resolver reuses its verification only while the file identity remains unchanged.

Missing or changed files raise an error. Do not replace this check with an unverified fallback or treat a metadata-only pass as a pixel check.

In [ ]:
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    from google.colab import drive
    drive.mount("/content/drive")

resolver = ImageResolver(original_roots=ORIGINAL_ROOTS, archive_root=ARCHIVE_ROOT,
                         derived_roots=DERIVED_ROOTS, archive_manifest=benchmark.archive_manifest,
                         source_records=source_records)
previews = []
write_json(WORK / "sample_report.json", {"status": "running", "training_ready": False})
try:
    for example in examples:
        print("Verifying", example["variant"], flush=True)
        previews.append(benchmark.prepare(example, resolver, PREVIEW_QUESTION))
except Exception as error:
    write_json(WORK / "sample_report.json", {"status": "failed", "training_ready": False,
                                            "error_type": type(error).__name__, "error": str(error)})
    raise


## 5. Display actual inputs and inspect the three separate contracts

In [ ]:
from IPython.display import display
for preview in previews:
    print(preview["provenance"]["variant"], "—", preview["model"]["image_roles"])
    for image in preview["model"]["images"]:
        display(image)

model_input = previews[0]["model"]          # Only this object may reach the model adapter.
evaluator = previews[0]["evaluator"]       # Annotation evidence; reference_answer is None.
provenance = previews[0]["provenance"]     # Source/product/annotation/example links.
print("Model keys:", list(model_input))
print("QA status:", evaluator["qa_validation_status"])
print("Training ready:", previews[0]["training_ready"])
# Optional task context must be explicitly allowlisted; the default is empty.
# benchmark.prepare(examples[0], resolver, PREVIEW_QUESTION,
#                   allowed_context=("task_instruction",),
#                   context={"task_instruction": "Describe visible morphology only."})


## 6. Save the verification record

In [ ]:
report = {"schema_version": 1, "status": "image_smoke_passed", "preview_only": True,
          "training_ready": False, "implementation_commit": CODE_COMMIT,
          "resolved_commit": store.resolve(),
          "samples": [{"example_id": p["provenance"]["example_id"],
                       "variant": p["provenance"]["variant"],
                       "image_count": len(p["model"]["images"]),
                       "image_sizes": [list(im.size) for im in p["model"]["images"]],
                       "pixels_verified_now": True} for p in previews]}
write_json(WORK / "sample_report.json", report)
write_json(WORK / "source_snapshot.json", store.snapshot())
write_json(WORK / "evaluator_records.json", [p["evaluator"] for p in previews])
write_json(WORK / "provenance.json", [p["provenance"] for p in previews])
print("Verified:", len(previews), "samples. Reports:", WORK)
print("Next: author and validate QA, assign group-safe splits, then add model/evaluator runners.")
